# Proyecto Oráculo · Notebook de evaluación (Grupo 1)

**Análisis y Diseño de Algoritmos · 2026-B · ADAG1**

Este notebook **evalúa la configuración entregada en `entrega.json` sobre los datos de prueba reservados a la profesora**. Está preparado para ejecutarse con *Ejecutar todo*, sin editar código, rutas, variables ni parámetros, y sin acceso previo a los datos privados.

**Qué hace, en orden**

1. Prepara el entorno (dependencias, verificadores, `oraculo.py`/`ayudas.py`) y localiza solos el repositorio y el archivo de prueba.
2. Carga y valida `entrega.json` (grupo, semana, índices 0–7 por ranura y temperatura).
3. Carga el modelo (`qwen17b`) y construye el oráculo con una **caché propia y vacía** (`cache_evaluacion_<huella>.json`), para no reutilizar respuestas de la búsqueda.
4. **Evalúa la configuración oficial** (`entrega.json`) sobre los datos de prueba: precisión, intervalo de confianza y familias de restricciones que más fallan.
5. **Compara las tres propuestas** del grupo (backtracking con poda, descenso de coordenadas y genético) en igualdad de condiciones: mismos datos, misma semilla, mismo modelo.
6. Guarda un resumen en `resultados_evaluacion.json`.

**Reglas de evaluación fijas:** temperatura 0.0 (decodificación *greedy*), `semilla=1`, puntaje todo-o-nada por instancia.

## 1 · Entorno, repositorio y verificadores

Esta celda no requiere reiniciar el entorno. Solo instala lo que falte, localiza la carpeta que contiene `oraculo.py` (si no la encuentra, descarga `oraculo.py`, `ayudas.py` y `datos_visibles.json` del repositorio del curso) y registra los verificadores de `open-instruct`. Intenta además registrar los de **IFBench**, necesarios para familias de restricciones que no están en `datos_visibles.json`.

In [ ]:
import importlib, importlib.util, io, json, os, subprocess, sys, urllib.request, zipfile
from pathlib import Path

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

# ── 1. Dependencias: solo se instala lo que falta (sin reinicio de entorno) ──
PAQUETES = ["nltk", "spacy", "emoji", "langdetect", "immutabledict", "accelerate", "transformers"]
faltan = [p for p in PAQUETES if importlib.util.find_spec(p) is None]
if faltan:
    print("instalando:", faltan)
    sh(f"{sys.executable} -m pip install -q --retries 1 --timeout 20 " + " ".join(faltan))
    importlib.invalidate_caches()
try:
    import spacy
    if not spacy.util.is_package("en_core_web_sm"):
        sh(f"{sys.executable} -m spacy download en_core_web_sm -q")
        importlib.invalidate_caches()
except Exception as e:
    print("aviso (spacy):", e)

# ── 2. Datos de nltk (descarga manual: el downloader rechaza el proxy de Colab) ──
NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
for carpeta, nombre in [("tokenizers", "punkt"), ("tokenizers", "punkt_tab"),
                        ("taggers", "averaged_perceptron_tagger"),
                        ("taggers", "averaged_perceptron_tagger_eng")]:
    destino = Path.home() / "nltk_data" / carpeta
    if (destino / nombre).exists():
        continue
    try:
        with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip", timeout=30) as resp:
            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(destino)
    except Exception as e:
        print(f"aviso (nltk {nombre}):", e)

# ── 3. Verificadores de open-instruct: oraculo.py los busca en /content/open-instruct ──
os.makedirs("/content", exist_ok=True)
if not Path("/content/open-instruct").exists():
    sh("git clone -q https://github.com/allenai/open-instruct /content/open-instruct")

# ── 4. Localizar la raíz del repositorio (la carpeta que contiene oraculo.py) ──
AQUI = Path.cwd().resolve()
CANDIDATAS_RAIZ = [AQUI, AQUI.parent, AQUI.parent.parent, Path("/content")]
CANDIDATAS_RAIZ += [p for p in sorted(Path("/content").glob("*")) if p.is_dir()]
REPO_GRUPO = "https://github.com/juanchito1234/proyecto-analisis-oraculo"
RAIZ = next((p for p in CANDIDATAS_RAIZ if (p / "oraculo.py").exists() and (p / "entrega.json").exists()), None)
if RAIZ is None:      # el notebook se abrió solo (p. ej. desde GitHub): se trae el repositorio del grupo
    destino = Path("/content/proyecto-analisis-oraculo")
    if not destino.exists():
        sh(f"git clone -q {REPO_GRUPO} {destino}")
    if (destino / "entrega.json").exists():
        RAIZ = destino
        print("notebook abierto fuera del repositorio: se clonó", REPO_GRUPO)
if RAIZ is None:      # último recurso: solo archivos del curso (sin entrega.json; la celda 3 avisará)
    RAIZ = next((p for p in CANDIDATAS_RAIZ if (p / "oraculo.py").exists()), None)
if RAIZ is None:
    RAIZ = AQUI
    REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-Oraculo-AyD/main"
    for f in ["oraculo.py", "ayudas.py", "datos_visibles.json"]:
        urllib.request.urlretrieve(f"{REPO}/{f}", RAIZ / f)
    print("oraculo.py no estaba en el repositorio: descargado del repositorio del curso")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
print("raíz del repositorio:", RAIZ)

# ── 5. Verificadores extra (IFBench). Si no se pueden cargar, se avisa y se sigue ──
REGISTROS_EXTRA = []
try:
    if not Path("/content/IFBench").exists():
        sh("git clone -q https://github.com/allenai/IFBench /content/IFBench")
    if Path("/content/IFBench").exists():
        sys.path.append("/content/IFBench")
        import instructions_registry as _ifbench
        REGISTROS_EXTRA.append(_ifbench.INSTRUCTION_DICT)
except Exception as e:
    print("aviso: no se cargó el registro de IFBench:", repr(e))
print("registros extra cargados:", len(REGISTROS_EXTRA))

## 2 · Configuración entregada (`entrega.json`) y comparación

`entrega.json` es la **única fuente** de la configuración oficial. La celda lo valida con las mismas reglas de la interfaz del proyecto: JSON válido, `grupo` y `semana` presentes, un entero entre 0 y 7 por ranura y `temperatura` ∈ {0.0, 0.3, 0.7}.

Además carga las configuraciones que produjo cada algoritmo (carpeta `entregas/`) para compararlas en la sección 5, y avisa (sin detener la corrida) si la oficial no coincide con la del notebook de búsqueda de backtracking (`entregas/entrega_backtracking_poda.json`).

In [ ]:
from oraculo import CATALOGO, RANURAS, TEMPERATURAS

def ubicar(nombre, subcarpeta=""):
    for base in [RAIZ, AQUI, AQUI.parent, Path("/content")]:
        p = base / subcarpeta / nombre
        if p.exists():
            return p
    return None

def cargar_entrega(ruta):
    d = json.load(open(ruta))
    errores = [f"falta la llave '{k}'" for k in ("grupo", "config", "semana") if k not in d]
    cfg = d.get("config", {})
    for r in RANURAS:
        v = cfg.get(r)
        if not (isinstance(v, int) and not isinstance(v, bool) and 0 <= v < len(CATALOGO[r])):
            errores.append(f"'{r}' debe ser un entero entre 0 y {len(CATALOGO[r]) - 1} (vale {v!r})")
    t = cfg.get("temperatura")
    if isinstance(t, bool) or not isinstance(t, (int, float)) or float(t) not in TEMPERATURAS:
        errores.append(f"'temperatura' debe ser uno de {TEMPERATURAS} (vale {t!r})")
    if errores:
        raise ValueError(f"{ruta}: entrega inválida → " + "; ".join(errores))
    cfg = {r: int(cfg[r]) for r in RANURAS} | {"temperatura": float(cfg["temperatura"])}
    return {"grupo": d["grupo"], "semana": d["semana"], "config": cfg}

RUTA_ENTREGA = ubicar("entrega.json")
assert RUTA_ENTREGA is not None, "No se encontró entrega.json en el repositorio"
ENTREGA = cargar_entrega(RUTA_ENTREGA)
CONFIG_OFICIAL = ENTREGA["config"]
print(f"entrega.json → {RUTA_ENTREGA}")
print(f"grupo: {ENTREGA['grupo']} · semana: {ENTREGA['semana']}")
print("configuración oficial:", CONFIG_OFICIAL)

# Configuraciones de cada algoritmo. Si falta un archivo se usa la reportada en su notebook.
RESPALDO = {
    "Backtracking con poda":    dict(rol=6, estrategia=2, formato=0, estilo=6, cierre=3, temperatura=0.0),
    "Descenso de coordenadas":  dict(rol=6, estrategia=2, formato=0, estilo=6, cierre=3, temperatura=0.0),
    "Genético":                 dict(rol=4, estrategia=3, formato=7, estilo=5, cierre=0, temperatura=0.0),
}
ARCHIVOS = {"Backtracking con poda": "entrega_backtracking_poda.json",
            "Descenso de coordenadas": "entrega_descenso_coordenadas.json",
            "Genético": "entrega_genetico.json"}
PROPUESTAS = {}
for nombre, archivo in ARCHIVOS.items():
    ruta = ubicar(archivo, "entregas")
    PROPUESTAS[nombre] = cargar_entrega(ruta)["config"] if ruta else RESPALDO[nombre]
    print(f"{nombre:26s} ← {ruta if ruta else 'respaldo embebido (no se halló ' + archivo + ')'}")

# Coherencia: entrega.json debe ser la configuración que justifica el notebook de búsqueda.
if CONFIG_OFICIAL == PROPUESTAS["Backtracking con poda"]:
    print("\n✔ entrega.json coincide con la configuración justificada en el notebook de búsqueda (backtracking).")
else:   # solo aviso: una incoherencia interna no debe impedir la evaluación
    print("\n⚠ entrega.json NO coincide con entregas/entrega_backtracking_poda.json; se evalúa entrega.json.")

## 3 · Datos de prueba reservados

El archivo de prueba se **detecta por su estructura**, sin pedir ruta: se acepta cualquier `.json` en el repositorio (hasta dos carpetas de profundidad) o en `/content` cuyo contenido sea una lista de instancias (o una sola, o `{"clave": [lista]}`, o JSON Lines) con los campos `prompt` e `ids` (mismo formato que `datos_visibles.json`), excluyendo `datos_visibles.json`, cachés, registros y entregas. Si `datos_visibles.json` no trae el campo `split`, se entiende que fue reemplazado por los datos de prueba y se evalúa completo. Si hay varios candidatos, se prefiere el que lleve en el nombre `prueba`, `test`, `privad`, `oculto` o `reservado`.

Si una instancia pide una familia de restricciones que ningún verificador cargado conoce, **no se detiene la corrida**: se excluye del puntaje y se informa en voz alta cuántas fueron (para no inflar ni deformar el resultado en silencio).

> **Modo ensayo.** Si no hay ningún archivo de prueba, el notebook usa las 87 instancias de validación visibles para comprobar que todo funciona, y lo declara con un aviso. Esos resultados **no** son la evaluación final.

In [ ]:
from collections import Counter

IGNORAR_NOMBRES = ("datos_visibles", "cache", "registro", "entrega", "resultados")
IGNORAR_CARPETAS = {"open-instruct", "IFBench", "drive", "sample_data", ".git", "node_modules"}
PREFERIDOS = ("prueba", "test", "privad", "oculto", "reservado")

def como_dataset(ruta):
    try:
        d = json.load(open(ruta))
    except Exception:
        return None
    if isinstance(d, list) and d and isinstance(d[0], dict) and {"prompt", "ids"} <= set(d[0]):
        return d
    return None

def buscar_privados():
    vistos, hallados = set(), []
    for base in [RAIZ, AQUI, AQUI.parent, Path("/content")]:
        patrones = ("*.json", "*/*.json", "*/*/*.json") if base == RAIZ else ("*.json", "*/*.json")
        for patron in patrones:
            for p in sorted(base.glob(patron)):
                if p.resolve() in vistos or any(x in p.name.lower() for x in IGNORAR_NOMBRES):
                    continue
                if IGNORAR_CARPETAS & set(p.parts):
                    continue
                vistos.add(p.resolve())
                d = como_dataset(p)
                if d:
                    hallados.append((p, d))
    hallados.sort(key=lambda h: (not any(k in h[0].name.lower() for k in PREFERIDOS), str(h[0])))
    return hallados

hallados = buscar_privados()
MODO_ENSAYO = not hallados
if hallados:
    ruta_prueba, datos_prueba = hallados[0]
    print("datos de prueba:", ruta_prueba, f"({len(datos_prueba)} instancias)")
    if len(hallados) > 1:
        print("otros candidatos ignorados:", [str(h[0]) for h in hallados[1:]])
    if all("split" in x for x in datos_prueba):          # si traen partición, se excluyen las descartadas
        datos_prueba = [x for x in datos_prueba if x["split"] != "descartada"]
else:
    from ayudas import cargar_datos, dividir
    visibles = cargar_datos(str(RAIZ / "datos_visibles.json"))
    if all("split" in x for x in visibles):
        _, datos_prueba = dividir(visibles)
        print("\n" + "!" * 70)
        print("MODO ENSAYO: no se encontró el archivo de prueba reservado.")
        print("Se usan las instancias de VALIDACIÓN visibles solo para verificar el notebook.")
        print("Carpetas revisadas:", sorted({str(b) for b in [RAIZ, AQUI, AQUI.parent, Path('/content')]}))
        print("!" * 70)
    else:   # datos_visibles.json fue reemplazado por los datos de prueba (no trae 'split')
        datos_prueba, MODO_ENSAYO = visibles, False
        print("datos_visibles.json no trae 'split': se usa completo como datos de evaluación.")

# Cada instancia necesita un 'id' (la caché lo usa en su clave).
datos_prueba = [dict(x, id=str(x.get("id", f"prueba_{i}"))) for i, x in enumerate(datos_prueba)]
for x in datos_prueba:                                    # 'ids' y 'kwargs' deben ser listas
    for k in ("ids", "kwargs"):
        if isinstance(x.get(k), str):
            x[k] = json.loads(x[k].replace("'", '"'))
print(f"instancias de evaluación: {len(datos_prueba)}")

## 4 · Modelo y oráculo

Mismo modelo que en la búsqueda (`qwen17b` = `Qwen/Qwen3-1.7B`, fp16, requiere GPU T4 en Colab), para que la evaluación sea comparable. La caché es **nueva** (`cache_evaluacion_<huella>.json` (la huella depende de los datos evaluados)): los ids de las instancias de prueba podrían coincidir con ids de las instancias visibles, y reusar la caché de la búsqueda mezclaría respuestas de otras instancias.

In [ ]:
from ayudas import cargar_modelo
from oraculo import Oraculo

modelo = cargar_modelo("qwen17b")
import hashlib
huella = hashlib.md5(json.dumps([[x["id"], x["prompt"], x.get("restriccion", "")] for x in datos_prueba],
                                sort_keys=True).encode()).hexdigest()[:8]
oraculo = Oraculo(modelo, datos_prueba, None, cache=str(Path.cwd() / f"cache_evaluacion_{huella}.json"),
                  lote=16, presupuesto=32_000_000, registros=REGISTROS_EXTRA)

# Familias que ningún verificador conoce: se excluyen del puntaje y se reportan.
def conocida(x):
    try:
        for i in x["ids"]:
            oraculo._clase(i)
        return True
    except KeyError:
        return False

sin_verificador = [x for x in datos_prueba if not conocida(x)]
if sin_verificador:
    fam = sorted({i for x in sin_verificador for i in x["ids"] if not conocida({"ids": [i]})})
    print(f"⚠ {len(sin_verificador)} instancias usan familias sin verificador cargado: {fam}")
    print("  Se excluyen del puntaje (cargar su registro en REGISTROS_EXTRA las incluiría).")
EVAL = [x for x in datos_prueba if conocida(x)]
print(f"instancias evaluables: {len(EVAL)} de {len(datos_prueba)}")
assert EVAL, "Ninguna instancia tiene verificador disponible: revise el registro de verificadores (celda 1)."

## 5 · Evaluación de la configuración oficial (`entrega.json`)

Una sola llamada a `oraculo.evaluar` con `semilla=1`. Se reporta la precisión (fracción de instancias que cumplen **todas** sus restricciones), un intervalo de confianza de Wilson del 95 % (con pocas instancias la incertidumbre es grande) y las familias que más veces tumbaron la respuesta.

In [ ]:
import math

def wilson(aciertos, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = aciertos / n
    d = 1 + z * z / n
    centro = (p + z * z / (2 * n)) / d
    mitad = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, centro - mitad), min(1.0, centro + mitad)

def resumir(config, etiqueta):
    r = oraculo.evaluar(config, EVAL, semilla=1, desc=etiqueta)
    aciertos = r.n - len(r.trazas)
    lo, hi = wilson(aciertos, r.n)
    return r, aciertos, lo, hi

r_of, ac_of, lo_of, hi_of = resumir(CONFIG_OFICIAL, "entrega.json")
print("=" * 66)
print(f"CONFIGURACIÓN OFICIAL: {CONFIG_OFICIAL}")
print(f"Precisión en {'ENSAYO (validación visible)' if MODO_ENSAYO else 'datos de prueba'}: "
      f"{r_of.precision:.1%}  ({ac_of}/{r_of.n})")
print(f"IC 95 % (Wilson): [{lo_of:.1%}, {hi_of:.1%}]")
print("=" * 66)
print("Familias que más fallaron (primera restricción violada):")
for fam, veces in Counter(t["violo"] for t in r_of.trazas).most_common(8):
    print(f"  {veces:3d}  {fam}")

## 6 · Comparación de las tres propuestas en igualdad de condiciones

Las tres configuraciones se miden sobre **las mismas instancias**, con el mismo modelo y la misma semilla. La tabla incluye, como referencia, lo que cada notebook de búsqueda reportó (precisión sobre su lote de búsqueda y sobre las 87 instancias de validación). Esas cifras **no** son comparables entre sí —cada algoritmo buscó con un lote y un punto de partida distintos—; la columna de prueba sí lo es.

Con un número moderado de instancias, diferencias de pocos puntos no son estadísticamente distinguibles: por eso se agrega la prueba exacta de Fisher entre cada propuesta y la mejor.

In [ ]:
import pandas as pd

# Cifras reportadas por los notebooks de búsqueda (referencia, no se recalculan aquí).
REFERENCIA = {   # (precisión búsqueda, instancias de búsqueda, precisión validación n=87)
    "Backtracking con poda":   (0.200, 100, 0.230),
    "Descenso de coordenadas": (0.200, 100, 0.230),
    "Genético":                (0.160,  25, 0.092),
}

filas, detalle = [], {}
for nombre, cfg in PROPUESTAS.items():
    r, ac, lo, hi = resumir(cfg, nombre)
    detalle[nombre] = (r, ac)
    bus, nb, val = REFERENCIA[nombre]
    filas.append({"algoritmo": nombre, "config": {k: cfg[k] for k in RANURAS},
                  "aciertos": ac, "n": r.n, "precision_prueba": r.precision,
                  "IC95_inf": lo, "IC95_sup": hi,
                  "ref_busqueda": bus, "ref_n_busqueda": nb, "ref_validacion87": val})

tabla = pd.DataFrame(filas).sort_values("precision_prueba", ascending=False).reset_index(drop=True)
vista = tabla.copy()
for c in ("precision_prueba", "IC95_inf", "IC95_sup", "ref_busqueda", "ref_validacion87"):
    vista[c] = vista[c].map("{:.1%}".format)
display(vista)

mejor_nombre = tabla.loc[0, "algoritmo"]
print(f"\nMayor precisión en {'ensayo' if MODO_ENSAYO else 'prueba'}: {mejor_nombre}")
try:
    from scipy.stats import fisher_exact
    _, ac_m = detalle[mejor_nombre]
    n_m = detalle[mejor_nombre][0].n
    for nombre, (r, ac) in detalle.items():
        if nombre != mejor_nombre:
            p = fisher_exact([[ac_m, n_m - ac_m], [ac, r.n - ac]])[1]
            print(f"  {mejor_nombre} vs {nombre}: p (Fisher exacta, dos colas) = {p:.3f}")
except Exception as e:
    print("(prueba de Fisher no disponible:", e, ")")

if CONFIG_OFICIAL != PROPUESTAS[mejor_nombre]:
    print("\n⚠ La configuración de entrega.json NO es la de mayor precisión en esta evaluación.")
else:
    print("\n✔ La configuración de entrega.json es la de mayor precisión en esta evaluación.")
iguales = [n for n, c in PROPUESTAS.items() if c == CONFIG_OFICIAL]
print("Propuestas cuya configuración coincide con la oficial:", ", ".join(iguales))


## 7 · Resumen y archivo de resultados

Se guarda `resultados_evaluacion.json` con la configuración oficial, el número de instancias evaluadas y la precisión de cada propuesta. Recuerde: la precisión medida aquí es la **mejor configuración encontrada**, no una garantía de optimalidad global del espacio de 32 768 configuraciones.

In [ ]:
resumen = {
    "grupo": ENTREGA["grupo"], "semana": ENTREGA["semana"],
    "modo_ensayo": MODO_ENSAYO,
    "instancias_evaluadas": len(EVAL),
    "instancias_sin_verificador": len(sin_verificador),
    "configuracion_oficial": CONFIG_OFICIAL,
    "precision_oficial": r_of.precision,
    "ic95_oficial": [lo_of, hi_of],
    "propuestas": {
        fila["algoritmo"]: {"config": fila["config"], "aciertos": int(fila["aciertos"]),
                            "n": int(fila["n"]), "precision": float(fila["precision_prueba"])}
        for fila in filas
    },
}
with open("resultados_evaluacion.json", "w") as f:
    json.dump(resumen, f, indent=1, ensure_ascii=False)
print(json.dumps(resumen, indent=1, ensure_ascii=False))
if MODO_ENSAYO:
    print("\n" + "!" * 70 + "\nRESULTADO DE ENSAYO (datos de validación visibles): no es la evaluación final.\n" + "!" * 70)
